# 🫁 Physiological Deepfake Detection: Cross-Dataset Generalization (Google Colab)

This notebook evaluates the **zero-shot cross-dataset generalization** of physiological and graph-consistency deepfake detectors. Specifically, we will:
1. Install required dependencies (MediaPipe, OpenCV, etc.).
2. Load pre-extracted features from the **SDFVD** dataset and train our best physiological models.
3. Load/Upload a subset of another dataset (like **Celeb-DF** or **FaceForensics++**).
4. Extract physiological features from the target dataset.
5. Evaluate how well the models trained on SDFVD generalize to the target dataset without any retraining (zero-shot transfer).

## 🚀 Step 1: Install Dependencies & Setup Environment

In [ ]:
# Install required libraries
!pip install -q mediapipe opencv-python scikit-learn numpy scipy tqdm matplotlib

# Download the MediaPipe Face Landmarker model asset
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
print("✅ Environment setup complete. Downloaded face_landmarker.task.")

## 📥 Step 1.5: Download External Datasets (Celeb-DF & FaceForensics++)
We can download Celeb-DF and FaceForensics++ directly into Google Colab via Kaggle API download endpoints using curl.

In [ ]:
# Create download directory
!mkdir -p /content/downloads

# Download FaceForensics++ (FF-C23) zip file
!curl -L -o /content/downloads/ff-c23.zip \
  https://www.kaggle.com/api/v1/datasets/download/xdxd003/ff-c23

# Download Celeb-DF-v2 zip file
!curl -L -o /content/downloads/celeb-df-v2.zip \
  https://www.kaggle.com/api/v1/datasets/download/reubensuju/celeb-df-v2

print("✅ Download commands executed. Check /content/downloads/ for zip files.")

### Unzip the downloaded datasets
Run this cell to extract the downloaded datasets.

In [ ]:
import os
# Extract Celeb-DF-v2
if os.path.exists('/content/downloads/celeb-df-v2.zip'):
    print("📦 Extracting Celeb-DF-v2...")
    !mkdir -p /content/celebdf_extracted
    !unzip -q -o /content/downloads/celeb-df-v2.zip -d /content/celebdf_extracted
    print("✅ Celeb-DF-v2 extracted!")

# Extract FaceForensics++
if os.path.exists('/content/downloads/ff-c23.zip'):
    print("📦 Extracting FaceForensics++...")
    !mkdir -p /content/ff_extracted
    !unzip -q -o /content/downloads/ff-c23.zip -d /content/ff_extracted
    print("✅ FaceForensics++ extracted!")

## 📁 Step 2: Upload Pre-Extracted SDFVD Features
To train the base models, upload the `graph_consistency_features.npz` file from your local machine. If you cloned the repository in Colab, you can find it under `deepfake-detection-physio/results/graph_consistency_features.npz`.

In [ ]:
import os
from google.colab import files

npz_path = 'graph_consistency_features.npz'

# If the repository was cloned, copy it to the local workspace
repo_path = 'deepfake-detection-physio/results/graph_consistency_features.npz'
if os.path.exists(repo_path):
    !cp {repo_path} .
    print("✅ Found graph_consistency_features.npz in the cloned repository!")
elif not os.path.exists(npz_path):
    print("📂 Please upload your 'graph_consistency_features.npz' file:")
    uploaded = files.upload()
else:
    print("✅ graph_consistency_features.npz is already present!")

## 🔬 Step 3: Implement Feature Extractors
This cell implements the physiological signal extractor (`EnhancedPhysioFeatureExtractor`) and the Physiological Consistency Graph Network extractor (`PCGNFeatureExtractor`).

In [ ]:
import cv2
import numpy as np
import re
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from scipy import stats, signal

def compute_max_cross_correlation(s1, s2, max_lag=15):
    s1_norm = (s1 - np.mean(s1)) / (np.std(s1) + 1e-7)
    s2_norm = (s2 - np.mean(s2)) / (np.std(s2) + 1e-7)
    best_corr = 0.0
    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            c = np.mean(s1_norm[-lag:] * s2_norm[:lag])
        elif lag > 0:
            c = np.mean(s1_norm[:-lag] * s2_norm[lag:])
        else:
            c = np.mean(s1_norm * s2_norm)
        if abs(c) > abs(best_corr):
            best_corr = c
    return best_corr

class EnhancedPhysioFeatureExtractor:
    def __init__(self, model_path="face_landmarker.task"):
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            output_face_blendshapes=True,
            num_faces=1,
        )
        self.landmarker = vision.FaceLandmarker.create_from_options(options)

    def extract_rppq_features(self, rgb_signals, fps):
        if len(rgb_signals) < 60:
            return [0.0] * 8
        rgb = np.array(rgb_signals)
        r_norm = (rgb[:, 0] - np.mean(rgb[:, 0])) / (np.std(rgb[:, 0]) + 1e-7)
        g_norm = (rgb[:, 1] - np.mean(rgb[:, 1])) / (np.std(rgb[:, 1]) + 1e-7)
        b_norm = (rgb[:, 2] - np.mean(rgb[:, 2])) / (np.std(rgb[:, 2]) + 1e-7)
        X = 3 * r_norm - 2 * g_norm
        Y = 1.5 * r_norm + g_norm - 1.5 * b_norm
        nyquist = 0.5 * fps
        low, high = 0.7 / nyquist, 4.0 / nyquist
        try:
            b_coeff, a_coeff = signal.butter(4, [low, high], btype='band')
            X_f = signal.filtfilt(b_coeff, a_coeff, X)
            Y_f = signal.filtfilt(b_coeff, a_coeff, Y)
            alpha = np.std(X_f) / (np.std(Y_f) + 1e-7)
            S = X_f - alpha * Y_f
            n = len(S)
            freqs = np.fft.fftfreq(n, 1 / fps)[:n // 2]
            magnitudes = np.abs(np.fft.fft(S)[:n // 2])
            valid_mask = (freqs >= 0.7) & (freqs <= 4.0)
            heart_rate = freqs[valid_mask][np.argmax(magnitudes[valid_mask])] * 60 if valid_mask.any() else 0.0
        except:
            heart_rate = 0.0
        features = [heart_rate, 1.0 if 40 <= heart_rate <= 180 else 0.0]
        if heart_rate > 40:
            peak_interval = 60.0 / heart_rate * fps
            peaks = np.arange(int(peak_interval), len(S), int(peak_interval))
            if len(peaks) > 2:
                intervals = np.diff(peaks) / fps * 1000
                rmssd = np.sqrt(np.mean(np.diff(intervals) ** 2))
                features.extend([rmssd, np.std(intervals), np.mean(np.abs(np.diff(intervals)) > 50) * 100, np.mean(intervals), np.std(intervals)])
            else:
                features.extend([0.0] * 5)
        else:
            features.extend([0.0] * 5)
        while len(features) < 8: features.append(0.0)
        return features

    def extract_blink_features(self, ear_signals, fps):
        if len(ear_signals) < 30:
            return [0.0] * 10
        ear = np.array(ear_signals)
        blinks = ear < 0.2
        blink_times = np.where(np.diff(blinks.astype(int)) == 1)[0]
        blink_rate = 60.0 / np.mean(np.diff(blink_times) / fps) if len(blink_times) >= 2 else 0.0
        features = [blink_rate, 1.0 if 5 <= blink_rate <= 40 else 0.0]
        features.extend([np.mean(ear), np.std(ear), np.min(ear), np.max(ear)])
        ear_diff = np.abs(np.diff(ear))
        features.extend([np.mean(ear_diff), np.std(ear_diff), np.max(ear_diff), np.sum(ear_diff > 0.05), np.var(ear)])
        while len(features) < 10: features.append(0.0)
        return features

    def extract_breathing_features(self, nose_positions, fps):
        if len(nose_positions) < 30:
            return [0.0] * 7
        nose = np.array(nose_positions)
        displacements = np.diff(nose[:, 1])
        fft_vals = np.fft.fft(displacements)
        freqs = np.fft.fftfreq(len(displacements), 1 / fps)
        valid = (freqs > 0.1) & (freqs < 1)
        breathing_rate = freqs[valid][np.argmax(np.abs(fft_vals[valid]))] * 60 if np.any(valid) else 0.0
        features = [breathing_rate, 1.0 if 5 <= breathing_rate <= 30 else 0.0]
        features.extend([np.std(displacements), np.max(np.abs(displacements)), np.mean(np.abs(displacements)), np.sum(np.abs(displacements) > np.std(displacements))])
        while len(features) < 7: features.append(0.0)
        return features

    def extract_color_features(self, rgb_signals):
        if len(rgb_signals) < 10:
            return [0.0] * 10
        rgb = np.array(rgb_signals)
        features = [np.std(rgb[:, 0]), np.std(rgb[:, 1]), np.std(rgb[:, 2])]
        gr_ratio = np.mean(rgb[:, 1]) / (np.mean(rgb[:, 0]) + 1e-7)
        features.extend([gr_ratio, 1.0 / (np.std(rgb) + 1e-7)])
        first_half = np.mean(rgb[:len(rgb)//2], axis=0)
        second_half = np.mean(rgb[len(rgb)//2:], axis=0)
        features.append(np.mean(np.abs(first_half - second_half)))
        features.append(np.std(rgb[:, 1]))
        skew = np.abs(stats.skew(rgb[:, 0])) + np.abs(stats.skew(rgb[:, 1])) + np.abs(stats.skew(rgb[:, 2]))
        features.append(skew)
        brightness = 0.299 * rgb[:, 0] + 0.587 * rgb[:, 1] + 0.114 * rgb[:, 2]
        features.extend([np.std(brightness), 1.0 / (np.std(brightness) + 1e-7)])
        while len(features) < 10: features.append(0.0)
        return features

    def extract_micro_movement_features(self, lm_seq, face_pos):
        features = []
        if len(face_pos) < 10:
            return [0.0] * 11
        face_arr = np.array(face_pos)
        displacements = np.diff(face_arr, axis=0)
        displacement_mag = np.linalg.norm(displacements, axis=1)
        features.extend([np.mean(displacement_mag), np.std(displacement_mag), np.max(displacement_mag), np.sum(displacement_mag > np.std(displacement_mag))])
        features.append(len(lm_seq) / max(1, len(face_pos)))
        if len(face_pos) > 10:
            vert_motion = np.std(np.diff(face_arr[:, 1]))
            horiz_motion = np.std(np.diff(face_arr[:, 0]))
            features.append(vert_motion / (horiz_motion + 1e-7))
            noise = np.mean(np.abs(np.diff(face_arr, n=2)))
            features.append(1.0 / (noise + 1e-7))
            features.append(np.mean(np.abs(np.diff(face_arr, n=2))))
        else:
            features.extend([0.0] * 4)
        while len(features) < 11: features.append(0.0)
        return features

    def extract_features_from_video(self, video_path):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps < 1: fps = 30
        rgb_signals, ear_signals = [], []
        face_pos, nose_pos, lm_seq = [], [], []
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            detection_result = self.landmarker.detect(mp_image)
            
            if detection_result.face_landmarks:
                lms = detection_result.face_landmarks[0]
                h, w = frame.shape[:2]
                face_pos.append((np.mean([lm.x for lm in lms]) * w, np.mean([lm.y for lm in lms]) * h))
                
                fh_y, fh_x = int(lms[10].y * h), int(lms[10].x * w)
                roi_h, roi_w = int(0.05 * h), int(0.1 * w)
                fh = frame[max(0, fh_y):min(h, fh_y+roi_h), max(0, fh_x-roi_w//2):min(w, fh_x+roi_w//2)]
                if fh.size > 0: rgb_signals.append(np.mean(fh.reshape(-1, 3), axis=0))

                def get_ear(indices):
                    pts = [np.array([lms[i].x * w, lms[i].y * h]) for i in indices]
                    return (np.linalg.norm(pts[1] - pts[5]) + np.linalg.norm(pts[2] - pts[4])) / (2.0 * np.linalg.norm(pts[0] - pts[3]) + 1e-7)

                ear_signals.append((get_ear([33, 160, 158, 133, 153, 144]) + get_ear([362, 385, 387, 263, 373, 380])) / 2.0)
                nose_pos.append((lms[4].x * w, lms[4].y * h))
                lm_seq.append(lms)
        
        cap.release()
        if len(rgb_signals) < 30: return None
        
        base_features = []
        base_features.extend(self.extract_rppq_features(rgb_signals, fps))
        base_features.extend(self.extract_blink_features(ear_signals, fps))
        base_features.extend(self.extract_breathing_features(nose_pos, fps))
        base_features.extend(self.extract_color_features(rgb_signals))
        base_features.extend(self.extract_micro_movement_features(lm_seq, face_pos))
        return np.array(base_features)

class PCGNFeatureExtractor(EnhancedPhysioFeatureExtractor):
    def extract_graph_features(self, video_path):
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps < 1: fps = 30
        rgb_signals, ear_signals = [], []
        face_pos, nose_pos, lm_seq = [], [], []
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
            detection_result = self.landmarker.detect(mp_image)
            
            if detection_result.face_landmarks:
                lms = detection_result.face_landmarks[0]
                h, w = frame.shape[:2]
                face_pos.append((np.mean([lm.x for lm in lms]) * w, np.mean([lm.y for lm in lms]) * h))
                
                fh_y, fh_x = int(lms[10].y * h), int(lms[10].x * w)
                roi_h, roi_w = int(0.05 * h), int(0.1 * w)
                fh = frame[max(0, fh_y):min(h, fh_y+roi_h), max(0, fh_x-roi_w//2):min(w, fh_x+roi_w//2)]
                if fh.size > 0: rgb_signals.append(np.mean(fh.reshape(-1, 3), axis=0))

                def get_ear(indices):
                    pts = [np.array([lms[i].x * w, lms[i].y * h]) for i in indices]
                    return (np.linalg.norm(pts[1] - pts[5]) + np.linalg.norm(pts[2] - pts[4])) / (2.0 * np.linalg.norm(pts[0] - pts[3]) + 1e-7)

                ear_signals.append((get_ear([33, 160, 158, 133, 153, 144]) + get_ear([362, 385, 387, 263, 373, 380])) / 2.0)
                nose_pos.append((lms[4].x * w, lms[4].y * h))
                lm_seq.append(lms)
        
        cap.release()
        if len(rgb_signals) < 60: return None
        
        # Generate time-series
        # 1. Heartbeat proxy
        rgb = np.array(rgb_signals)
        g_chan = rgb[:, 1]
        nyquist = 0.5 * fps
        b_cardiac, a_cardiac = signal.butter(4, [0.7/nyquist, 4.0/nyquist], btype='band')
        cardiac_signal = signal.filtfilt(b_cardiac, a_cardiac, g_chan)
        
        # 2. Respiration proxy
        nose = np.array(nose_pos)
        b_resp, a_resp = signal.butter(4, [0.1/nyquist, 1.0/nyquist], btype='band')
        resp_signal = signal.filtfilt(b_resp, a_resp, nose[:, 1])
        
        # 3. Eye Aspect Ratio (EAR)
        ear_signal = np.array(ear_signals)
        
        # 4. Head Vertical Velocity
        face_arr = np.array(face_pos)
        head_signal = np.zeros(len(face_arr))
        head_signal[1:] = np.diff(face_arr[:, 1])
        
        signals = [cardiac_signal, resp_signal, ear_signal, head_signal]
        min_len = min(len(s) for s in signals)
        signals = [s[:min_len] for s in signals]
        
        # Compute cross-correlations (6 edges)
        edges = []
        for i in range(len(signals)):
            for j in range(i + 1, len(signals)):
                edges.append(compute_max_cross_correlation(signals[i], signals[j], max_lag=15))
                
        # Base features
        base_f = []
        base_f.extend(self.extract_rppq_features(rgb_signals, fps))
        base_f.extend(self.extract_blink_features(ear_signals, fps))
        base_f.extend(self.extract_breathing_features(nose_pos, fps))
        base_f.extend(self.extract_color_features(rgb_signals))
        base_f.extend(self.extract_micro_movement_features(lm_seq, face_pos))
        
        return np.array(base_f), np.array(edges)

class FaceProcessor:
    def __init__(self, model_path='face_landmarker.task'):
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            num_faces=1,
        )
        self.landmarker = vision.FaceLandmarker.create_from_options(options)

    def detect_and_crop_face(self, frame, padding=0.2):
        h, w = frame.shape[:2]
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
        results = self.landmarker.detect(mp_image)
        if results.face_landmarks:
            lms = results.face_landmarks[0]
            xs = [lm.x for lm in lms]
            ys = [lm.y for lm in lms]
            x, y = min(xs) * w, min(ys) * h
            box_w, box_h = (max(xs) - min(xs)) * w, (max(ys) - min(ys)) * h
            pad_x = int(box_w * padding)
            pad_y = int(box_h * padding)
            x1 = max(0, int(x - pad_x))
            y1 = max(0, int(y - pad_y))
            x2 = min(w, int(x + box_w + pad_x))
            y2 = min(h, int(y + box_h + pad_y))
            face = frame[y1:y2, x1:x2]
            if face.size > 0:
                return cv2.resize(face, (128, 128))
        return None

print("✅ Extractors and FaceProcessor defined!")

## 🏋️ Step 4: Train Physiological RF & PCGN on SDFVD
We load the SDFVD dataset features, and train the RandomForest models to obtain the pre-trained weights.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Load the cached features
data = np.load(npz_path)
X_base = data['X_base'] # (94, 47)
X_graph = data['X_graph'] # (94, 6)
y = data['y']

print(f"Loaded SDFVD base features: {X_base.shape}")
print(f"Loaded SDFVD graph features: {X_graph.shape}")

# 1. Base Model
scaler_base = StandardScaler()
X_base_scaled = scaler_base.fit_transform(X_base)
model_base = RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_split=3, random_state=42, class_weight='balanced')
model_base.fit(X_base_scaled, y)

# 2. PCGN Graph Only Model
scaler_graph = StandardScaler()
X_graph_scaled = scaler_graph.fit_transform(X_graph)
model_graph = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, class_weight='balanced')
model_graph.fit(X_graph_scaled, y)

# 3. PCGN Graph + Base Combined Model
X_comb = np.hstack([X_base, X_graph])
scaler_comb = StandardScaler()
X_comb_scaled = scaler_comb.fit_transform(X_comb)
model_comb = RandomForestClassifier(n_estimators=300, max_depth=10, random_state=42, class_weight='balanced')
model_comb.fit(X_comb_scaled, y)

print("✅ Models successfully trained on SDFVD dataset!")

## 📁 Step 5: Download/Upload Celeb-DF Sample Videos
To demonstrate evaluation, you can upload some sample real/fake videos from Celeb-DF, or clone a small selection. In this cell, we prepare directories for evaluation.

In [ ]:
import os
os.makedirs('celebdf_test/real', exist_ok=True)
os.makedirs('celebdf_test/fake', exist_ok=True)
print("📁 Created folders: celebdf_test/real and celebdf_test/fake. Upload your test videos here!")

## 🏃 Step 6: Run Cross-Dataset Extraction and Evaluation
Once you have uploaded videos to the folders, run this cell to extract features and test the trained models.

In [ ]:
import glob
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

pcgn_extractor = PCGNFeatureExtractor()

X_test_base, X_test_graph, y_test = [], [], []

# Process real videos
real_videos = glob.glob('celebdf_test/real/*.mp4')
print(f"📂 Processing {len(real_videos)} real videos...")
for v in tqdm(real_videos):
    res = pcgn_extractor.extract_graph_features(v)
    if res is not None:
        base_f, graph_f = res
        X_test_base.append(base_f)
        X_test_graph.append(graph_f)
        y_test.append(0)

# Process fake videos
fake_videos = glob.glob('celebdf_test/fake/*.mp4')
print(f"\n📂 Processing {len(fake_videos)} fake videos...")
for v in tqdm(fake_videos):
    res = pcgn_extractor.extract_graph_features(v)
    if res is not None:
        base_f, graph_f = res
        X_test_base.append(base_f)
        X_test_graph.append(graph_f)
        y_test.append(1)

if len(y_test) == 0:
    print("❌ No target videos processed. Please upload videos to celebdf_test/real and celebdf_test/fake first!")
else:
    X_test_base = np.array(X_test_base)
    X_test_graph = np.array(X_test_graph)
    y_test = np.array(y_test)
    
    # Scalings (using SDFVD scalers to prevent target distribution leakage)
    X_test_base_scaled = scaler_base.transform(X_test_base)
    X_test_graph_scaled = scaler_graph.transform(X_test_graph)
    X_test_comb = np.hstack([X_test_base, X_test_graph])
    X_test_comb_scaled = scaler_comb.transform(X_test_comb)
    
    print("\n" + "="*50)
    print("CROSS-DATASET GENERALIZATION EVALUATION")
    print("="*50)
    
    # 1. Base Model Evaluation
    prob_base = model_base.predict_proba(X_test_base_scaled)[:, 1]
    pred_base = (prob_base >= 0.5).astype(int)
    acc_base = accuracy_score(y_test, pred_base)
    auc_base = roc_auc_score(y_test, prob_base) if len(np.unique(y_test)) > 1 else 0.5
    print(f"📊 Physiological Base Model (47 features): Accuracy = {acc_base*100:.2f}%, AUC = {auc_base:.4f}")
    
    # 2. Graph Only Model Evaluation
    prob_graph = model_graph.predict_proba(X_test_graph_scaled)[:, 1]
    pred_graph = (prob_graph >= 0.5).astype(int)
    acc_graph = accuracy_score(y_test, pred_graph)
    auc_graph = roc_auc_score(y_test, prob_graph) if len(np.unique(y_test)) > 1 else 0.5
    print(f"📊 PCGN Graph Only (6 features):          Accuracy = {acc_graph*100:.2f}%, AUC = {auc_graph:.4f}")
    
    # 3. Combined Model Evaluation
    prob_comb = model_comb.predict_proba(X_test_comb_scaled)[:, 1]
    pred_comb = (prob_comb >= 0.5).astype(int)
    acc_comb = accuracy_score(y_test, pred_comb)
    auc_comb = roc_auc_score(y_test, prob_comb) if len(np.unique(y_test)) > 1 else 0.5
    print(f"📊 Graph + Base Combined (53 features):    Accuracy = {acc_comb*100:.2f}%, AUC = {auc_comb:.4f}")

## 🤖 Step 7: Zero-Shot Evaluation of Visual Deep Learning Models

In this step, we load the pre-trained deep learning baselines (saved under the `models/` directory in the cloned repository) and test their zero-shot generalization on the external dataset (e.g., Celeb-DF or FaceForensics++).

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import os
from tqdm import tqdm
from sklearn.metrics import accuracy_score, roc_auc_score

# Paths to pre-trained model files (relative to cloned repo root)
dl_models = {
    "Baseline CRNN (MobileNetV2)": "deepfake-detection-physio/models/original_crnn_model.keras",
    "Stacked CRNN (MobileNetV2 + Attn)": "deepfake-detection-physio/models/improved_crnn_best.keras",
    "EfficientNet + CRNN (Frozen Backbone)": "deepfake-detection-physio/models/final/deepfake_model_final.keras",
    "XceptionNet (Single Frame)": "deepfake-detection-physio/models/best_model.keras"
}

# Configuration
IMG_SIZE = 128
SEQ_LENGTH = 20

face_processor = FaceProcessor()

def extract_dl_sequence(video_path, seq_length=20):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < seq_length:
        cap.release()
        return None
    indices = np.linspace(0, total_frames - 1, seq_length, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: break
        face = face_processor.detect_and_crop_face(frame)
        if face is not None:
            frames.append(face)
    cap.release()
    if len(frames) == seq_length:
        return np.array(frames)
    return None

# Compile target evaluation paths
test_videos = real_videos + fake_videos
test_labels = [0] * len(real_videos) + [1] * len(fake_videos)

if len(test_videos) == 0:
    print("❌ No target videos to evaluate. Please download/extract datasets first!")
else:
    print(f"🧠 Found {len(test_videos)} target videos for zero-shot evaluation.")
    for name, path in dl_models.items():
        if os.path.exists(path):
            print(f"\n🧠 Loading model {name} from {path}...")
            try:
                model = load_model(path)
                probs = []
                for v in tqdm(test_videos, desc=f"Evaluating {name}"):
                    frames = extract_dl_sequence(v, seq_length=SEQ_LENGTH)
                    if frames is not None:
                        prob = model(np.expand_dims(frames, axis=0), training=False).numpy().flatten()[0]
                        probs.append(prob)
                    else:
                        probs.append(0.5) # Default to neutral probability on face-extraction failure
                
                probs = np.array(probs)
                preds = (probs >= 0.5).astype(int)
                acc = accuracy_score(test_labels, preds)
                auc = roc_auc_score(test_labels, probs) if len(np.unique(test_labels)) > 1 else 0.5
                print(f"📊 {name}: Accuracy = {acc*100:.2f}%, AUC = {auc:.4f}")
                tf.keras.backend.clear_session()
            except Exception as e:
                print(f"❌ Failed to evaluate model {name}: {e}")
        else:
            print(f"⚠️ Model file {path} not found. Ensure repository is cloned or files are uploaded.")